In [1]:
!pip install flask tensorflow numpy 

In [ ]:
from flask import Flask, request, jsonify
import numpy as np
import tensorflow as tf
import librosa
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

# ===== Initialize Flask app =====
app = Flask(__name__)
app.config['MAX_CONTENT_LENGTH'] = 5 * 1024 * 1024  # Allow up to 5 MB audio uploads

# ===== Bird class labels =====
classes = ['parrot', 'peacock', 'sparrow', 'crow']

# ===== Load the float32 TFLite model =====
interpreter = tf.lite.Interpreter(model_path="bird_model.tflite")
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("✅ Float32 TFLite model loaded successfully!")
print(f"Input shape: {input_details[0]['shape']}")
print(f"Output shape: {output_details[0]['shape']}")

@app.route('/predict', methods=['POST'])
def predict():
    try:
        # ===== Read raw binary data from ESP32 =====
        audio_bytes = request.data
        if not audio_bytes or len(audio_bytes) == 0:
            return jsonify({'error': 'No audio data received'}), 400

        # Convert bytes to numpy array (int16)
        audio_data = np.frombuffer(audio_bytes, dtype=np.int16)

        # ===== Normalize to range [-1, 1] =====
        audio_data = audio_data.astype(np.float32) / 32768.0

        sr = 16000  # Sample rate

        # ===== Ensure 2 seconds (32000 samples) =====
        if len(audio_data) < 32000:
            audio_data = np.pad(audio_data, (0, 32000 - len(audio_data)))
        elif len(audio_data) > 32000:
            audio_data = audio_data[:32000]

        # ===== Compute log-mel spectrogram =====
        mel = librosa.feature.melspectrogram(
            y=audio_data,
            sr=sr,
            n_fft=1024,
            hop_length=512,
            n_mels=40,
            power=2.0
        )
        log_mel = librosa.power_to_db(mel, ref=np.max)

        # Normalize same as training
        log_mel = (log_mel - log_mel.mean()) / (log_mel.std() + 1e-9)

        # Ensure shape (40, 63)
        if log_mel.shape[1] > 63:
            log_mel = log_mel[:, :63]
        elif log_mel.shape[1] < 63:
            pad_width = 63 - log_mel.shape[1]
            log_mel = np.pad(log_mel, ((0, 0), (0, pad_width)), mode='constant')

        # ===== Quantization Fix =====
        # Get quantization parameters from model
        scale, zero_point = input_details[0]['quantization']

        # Convert log_mel → int8 using model's quantization scale
        input_data = log_mel[np.newaxis, :, :, np.newaxis] / scale + zero_point
        input_data = np.clip(input_data, -128, 127).astype(np.int8)

        # ===== Run inference =====
        interpreter.set_tensor(input_details[0]['index'], input_data)
        interpreter.invoke()
        prediction = interpreter.get_tensor(output_details[0]['index'])

        # ===== Softmax + confidence =====
        prediction = tf.nn.softmax(prediction[0].astype(np.float32)).numpy()
        predicted_class_index = int(np.argmax(prediction))
        confidence = float(prediction[predicted_class_index])

        # ===== Result threshold =====
        if confidence < 0.3:
            result = {
                'predicted_class_index': -1,
                'predicted_class': 'Unknown',
            }
        else:
            result = {
                'predicted_class_index': predicted_class_index,
                'predicted_class': classes[predicted_class_index],
            }

        return jsonify(result)

    except Exception as e:
        return jsonify({'error': str(e)}), 500


# ===== Run Flask app =====
if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)


✅ Float32 TFLite model loaded successfully!
Input shape: [ 1 40 63  1]
Output shape: [1 4]
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://10.99.105.191:5000
Press CTRL+C to quit
